# Goal

Получение нового baseline на основе новой `WorldModel`, которая натренирована на расширенном датасете. Расширенный датасет включает уровни вплоть до 10-го. Так что `VisionHead` должен быть восприимчив ко всем наюансам

# TARGET_NOTEBOOK_FNAME

In [1]:
TARGET_NOTEBOOK_FNAME = '18n_ppo_tr_frostbite_07.ipynb'

# GRID_SEARCH_SPACE

In [2]:
# @launchit.collect
# GRID_SEARCH_SPACE = dict(
# )

# set_hyperparameters

In [3]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    import random
    HP.general.random_seed = random.randint(1, 100)
    HP.general.is_torch_deterministic = True
    HP.general.is_torch_compile = True
    HP.general.is_torch_amp = True
    
    HP.env.count = 32 
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6

    parent = optuna_trial.suggest_categorical('parent',[
        '18d_world_model_09:185', # ssim=0.9938, highest ssim, partial problems with recon0.1+recon3.0
        '18d_world_model_09:183', # ssim=0.9936, decent ssim, no apparent problems with recon0.1+recon3.0
    ])
    HP.vision_head.parent = dict(model=parent, weights=parent)
    HP.vision_head.is_trainable = False
    
    HP.encoder.parent = dict(model=parent, weights=parent)
    HP.encoder.is_trainable = True

    HP.agent.parent = None 
    HP.agent.sequence_length = 4
    HP.agent.action_plan_length = 10
    HP.agent.d_model = 256 
    HP.agent.transformer = dict(layers_count=3, heads_count=4, attention_backend=['EFFICIENT_ATTENTION', 'MATH'])
    HP.agent.is_trainable = True
    
    # Test params
    HP.test.env_rams = [
        'com.develorium.neurolab.frostbite_ram:level1:1:cls=none',
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none',
    ]
    HP.test.env_ram_patches = [
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_right_of_igloo', 'bear_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_left_of_igloo', 'bear_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_right_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_left_of_igloo'],
        ['no_score', 'last_life', 'full_igloo', 'temperature_20', 'bailey_near_center'], # *
    ]
    HP.test.break_on_level_passed = True
    
    # Training procedure params (PPO related) 
    HP.ppo.global_steps_count = 3_000_000 # total number of steps 
    HP.ppo.rollout_steps_count = 512 # how many steps to run in a single policy rolllout
    HP.ppo.rollout_env_rams = [
        'com.develorium.neurolab.frostbite_ram:level1:1:cls=none',
        'com.develorium.neurolab.frostbite_ram:level5:1:cls=none',
    ]
    HP.ppo.rollout_env_ram_patches = [
        ['no_score', 'last_life', 'full_igloo',            'temperature_10', 'bailey_right_at_the_igloo_door'], # 0 
        ['no_score', 'last_life', 'full_igloo',            'temperature_10', 'bailey_very_near_igloo_door'], # 1
        ['no_score', 'last_life', 'full_igloo',            'temperature_10', 'bailey_near_center'], # 2
        ['no_score', 'last_life', 'one_remaining_igloo',   'temperature_10', 'bailey_near_center'], # 3
        ['no_score', 'last_life', 'three_remaining_igloo', 'temperature_20', 'bailey_near_center', ], # 4
        ['no_score', 'last_life', 'half_igloo', 'bailey_near_center'], # 5
        ['no_score', 'last_life', 'half_igloo'], # 6

        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_right_of_igloo', 'bear_to_the_left_of_igloo'], # 7
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bailey_to_the_left_of_igloo', 'bear_to_the_left_of_igloo'], # 8
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_right_of_igloo'], # 9
        ['no_score', 'last_life', 'full_igloo', 'temperature_10', 'bear_chases_bailey_to_the_left_of_igloo'], # 10
        ['no_score', 'last_life', 'full_igloo', 'temperature_20', 'bailey_near_center'], # 11
    ]
    HP.ppo.rollout_env_stories = [
        '0;0:7',  # level 1 - teach agent to enter complete igloo
        '1;7:12', # level 5 - teach agent to enter complete blinking igloo
    ]
    
    HP.ppo.epochs_count = 2 
    HP.ppo.batch_size = 512 
    HP.ppo.learn_rate = 'const(0.00025)'
    HP.ppo.optimizer = 'AdamW'
    
    HP.ppo.value_loss = dict(coef=0.2)
    HP.ppo.entropy_loss = dict(coef='const(0.05)')
    HP.ppo.consistency_loss = dict(type='MSE', coef=0.1)
    HP.ppo.prediction_loss = dict(coef=0.1)
    
    HP.ppo.gamma = 0.997 # return discount factor gamma
    HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
    HP.ppo.clip_coef = 0.1 # the surrogate clipping coefficient
    HP.ppo.clip_vloss = True
    HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
    HP.ppo.target_kl = None # the target KL divergence threshold
    HP.ppo.norm_adv = True # Toggles advantages normalization
    return HP

# Results

Проверяем: новый baseline на новой `WorldModel` (расширенный датасет, уровни до 10-го) — какая из родителей лучше: `18d_world_model_09:185` или `18d_world_model_09:183`?

`31.1` — родитель сэмплируется (`suggest_categorical`): рука `:185` (ssim=0.9938, по HP-комментариям частичные проблемы с recon/0.1+recon/3.0) и рука `:183` (ssim=0.9936, без видимых проблем), frozen `vision_head` + trainable `encoder` из той же версии. `consistency_loss=MSE` (после `30.1`). Базовое сравнение — с `30.1` (родители `:40`, KLDiv) и `20.1` (родители `:40`, MSE, MSE-база). Остальные HP идентичны (агент с нуля, `sequence_length=4`, `action_plan_length=10`, 3M шагов, 32 env, `learn_rate=const(0.00025)`, `ent_coef=const(0.05)`, `vf_coef=0.2`, `prediction_coef=0.1`; шаг 1: уровень 1, истории `0;0:7` и `1;7:12`).

**Серии: `31.1` vs `30.1` vs `20.1`** (метрика — `test/levels_passed_mean`)

| Metric | Experiment `18n_study_31.1` (новая WM, MSE) | Experiment `18n_study_30.1` (:40, KLDiv) | Experiment `18n_study_20.1` (:40, MSE) | Statistical Result / Verdict |
| :--- | :--- | :--- | :--- | :--- |
| **Sample Size (n)** | 24 | 24 | 24 | — |
| **Mean Score** | 0.6272 | 0.6047 | 0.6185 | +0.0225 vs `30.1`, +0.0087 vs `20.1` (н/з) |
| **Best Value** | 0.796875 (`521`, `528`) | 0.79375 (`421`, `430`) | 0.8375 (`155`) | — |
| **Standard Deviation** | 0.0881 | 0.1027 | 0.0997 | шум меньше у `31.1` |
| **Shapiro-Wilk (p-value)** | 0.1066 (нормальное) | 0.0367 (не нормальное) | 0.0594 (нормальное) | распределение `31.1` ближе к нормальному |
| **Welch's t-test (p-value)** | — | — | — | **0.4189** (не значимо, `31.1` vs `30.1`) |
| **Mann-Whitney U (p-value)** | — | — | — | **0.2823** (не значимо) |
| **Cohen's d (Effect Size)** | — | — | — | **0.2355** (малый, в пользу `31.1`) |

**Руки внутри `31.1`: `:183` vs `:185`** (реальный вопрос серии)

| Metric | Рука `:183` (n=13) | Рука `:185` (n=11) | Statistical Result / Verdict |
| :--- | :--- | :--- | :--- |
| **Mean Score** | 0.6707 | 0.5759 | +0.0948 в пользу `:183` |
| **Best Value** | 0.796875 (`521`, `528`) | 0.671875 (`515`) | — |
| **Standard Deviation** | 0.0867 | 0.0588 | — |
| **Shapiro-Wilk (p-value)** | 0.1812 (нормальное) | 0.4738 (нормальное) | обе руки — нормальные |
| **Welch's t-test (p-value)** | — | — | **0.0046** (значимо, в пользу `:183`) |
| **Mann-Whitney U (p-value)** | — | — | **0.0105** (значимо) |
| **Cohen's d (Effect Size)** | — | — | **1.259** (большой, в пользу `:183`) |

Статистически `:183` заметно сильнее `:185` — все три теста значимы, размер эффекта большой (d≈1.26). Топ-триала серии (`521`, `528` — 0.796875, `525` — 0.7875) все в руке `:183`; лучший результат руки `:185` — `515` (0.671875), худшие в серии (`516`, `524` — 0.496875) тоже в руке `:185`.

По сериям: `31.1` vs `30.1`/`20.1` — статистически неотличимы (все три теста н/з, d≈0.24). Новая `WorldModel` на решающем метрике не теряет, но и не даёт прироста на шаге 1. Учтено, что до `30.1` за раз сменились два параметра (родители + `MSE`/`KLDiv`), но по вердикту `30.1` эти лоссы на шаге 1 неразличимы — дельту приписываем родителям в первую очередь.

Системная проблема `recon/0.1+recon/3.0`: в журнале только решающий метрик — **не проверено по журналу**. По HP-комментариям: у `:185` частичные проблемы, у `:183` — нет видимых проблем; вердикт по проблеме за Боссом.

**Выводы**
1) Родитель нового baseline — `18d_world_model_09:183`: рука `:183` значимо бьёт `:185` (Welch p=0.0046, MWU p=0.0105, Cohen's d≈1.26); лучшие триала серии `521`/`528` (0.796875) и `525` (0.7875) — все в руке `:183`.
2) На решающем метрике новая `WorldModel` (расширенный датасет, уровни до 10-го) статистически неотличима от старого `:40`-бейслайна (n/з, d≈0.24): регрессии нет, но и прироста на шаге 1 нет — эффект расширенного датасета (нюансы поздних уровней) должен проявляться на более сложных шагах.
3) Статус `recon/0.1+recon/3.0` по данному исследованию — «не проверено по журналу» (в журнале только решающий метрик); по HP-комментариям у `:183` проблем нет.
4) Следующий шаг: чемпионы руки `:183` (`521`, `528`, `525`) — PARENTS для шага 2 нового baseline.

# System

In [4]:
import os, sys, re, subprocess, json
import IPython 
import concurrent.futures as cf
from collections import namedtuple

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
from optuna.trial import TrialState

project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]

sys.path.append(os.path.join(project_root_path, 'lib'))

from logging_utils import *
from math_utils import *
from artifact_registry import *
import launchit
import launch_dispatcher
from autoincrement import Autoincrement

In [5]:
CONFIG = namedtuple('CONFIG', 
                    'project_root_uri, model_group_uri, project_root_path, subproject_name, subproject_path, run_path, ' + 
                    'target_notebook_fname, target_notebook_name, ' + 
                    'optuna_study_notebook_fname, optuna_study_name, optuna_study_serial, optuna_study_fname')(
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    model_group_uri=None,
    project_root_path=project_root_path,
    subproject_name=None,
    subproject_path=os.path.abspath('../..'),
    run_path=None,
    target_notebook_fname=os.path.join(os.path.abspath('../..'), TARGET_NOTEBOOK_FNAME),
    target_notebook_name=None,
    optuna_study_notebook_fname=None,
    optuna_study_name=None,
    optuna_study_serial=None,
    optuna_study_fname=None,
)

with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as connection_file:
    optuna_study_notebook_fname = json.load(connection_file).get('jupyter_session')
    optuna_study_name, _ = os.path.splitext(os.path.basename(optuna_study_notebook_fname))
    optuna_study_serial = re.match(r'\w+_([\d\.\w]+)', optuna_study_name).group(1)
    optuna_study_fname = os.path.join(os.path.dirname(optuna_study_notebook_fname), optuna_study_name + '.optuna')
    CONFIG = CONFIG._replace(optuna_study_notebook_fname=optuna_study_notebook_fname)
    CONFIG = CONFIG._replace(optuna_study_name=optuna_study_name)
    CONFIG = CONFIG._replace(optuna_study_serial=optuna_study_serial)
    CONFIG = CONFIG._replace(optuna_study_fname=optuna_study_fname)

target_notebook_name, _ = os.path.splitext(os.path.basename(TARGET_NOTEBOOK_FNAME))
CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.target_notebook_fname)))
CONFIG = CONFIG._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
CONFIG = CONFIG._replace(target_notebook_name=target_notebook_name)
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG._asdict()

{'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'subproject_name': '18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'target_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/18n_ppo_tr_frostbite_07.ipynb',
 'target_notebook_name': '18n_ppo_tr_frostbite_07',
 'optuna_study_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_31.1/18n_study_31.1.ipynb',
 'optuna_study_name': '18n_study_31.1',
 'optuna_study_serial': '31.1',
 'optuna_study_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_31.1/18n_study_31.1.optuna'}

In [6]:
LOG = Logging.get()
LOG.enable('syslog', False)
LOG.enable('stdout', False)
LOG.enable('verbose_stdout', True)

In [7]:
ARTIFACT_REGISTRY = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri)

In [8]:
def create_optuna_launch():
    model_version = int(Autoincrement.get(f'{CONFIG.model_group_uri}.{CONFIG.target_notebook_name}'))
    assert model_version > 0, model_version
    ARTIFACT_REGISTRY.register_component(CONFIG.target_notebook_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    # Prep docker launch
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.target_notebook_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL='TRAIN',
        OPTUNA_STUDY_FNAME=CONFIG.optuna_study_fname,
        OPTUNA_STUDY_NAME=CONFIG.optuna_study_name,
        OPTUNA_DECISIVE_METRIC='test/levels_passed_mean',
    )
    launch_fname = launchit.launchit(
        CONFIG.target_notebook_fname, 
        launch_serial=int(model_version),
        expandvars=expandvars, 
        make_py_file=False, 
        dir_name=CONFIG.run_path,
        collect_inds=['temp_config', 'optuna', 'initrd', 'build_docker_launch', 'optuna_run_docker_launch'],
        disable_inds=[],
    )
    return f'{CONFIG.target_notebook_name}:{model_version}', launch_fname

In [9]:
# Executed in a separate thread with GIL locked
def run_optuna_launch(launch_fname):
    # Run launch notebook locally, the latter will:
    # 1) sample values of hyperparameters from optuna study
    # 2) pack everything to docker launch (self-contained thing)
    # 3) run "docker_launch_run" cell which in turn will dispatch launch to cloud via launch_dispatcher
    # 4) collect result of a docker launch from cloud and update optuna study
    LOG(f'Launching "{launch_fname}"')
    
    subprocess.run(
        ['papermill', launch_fname, launch_fname, '--no-progress-bar'],
        capture_output=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    if os.path.exists(launch_fname + '.out'):
        with open(launch_fname + '.out', 'rt') as f:
            LOG(f.read())
    else:
        LOG(f'"{launch_fname}" completed with no output, probably failed')

## Unleash!

In [15]:
optuna_study = optuna.create_study(
    study_name=CONFIG.optuna_study_name,
    directions=['maximize'],
    storage=JournalStorage(JournalFileBackend(file_path=CONFIG.optuna_study_fname)),
    load_if_exists=True,
)
optuna_study.set_user_attr('STUDY_SERIAL', CONFIG.optuna_study_serial)
launches_count = 6
completed_launches_count = 0

with LOG.auto_log_level(logging.INFO):
    with cf.ThreadPoolExecutor(max_workers=32) as executor:
        futures = {}
        idle_runners_af = RecursiveMovingAverageFilter(max_n=6)
        is_first_time = True
        
        while launches_count is None or completed_launches_count < launches_count:
            runners_info = launch_dispatcher.RunnersInfo.get()
            idle_runners_af(runners_info['idle'])

            if is_first_time or (idle_runners_af.n >= idle_runners_af.max_n and idle_runners_af.v >= 1):
                if launches_count is None or (completed_launches_count + len(futures) < launches_count):
                    launch_name, launch_fname = create_optuna_launch()
                    futures.update({executor.submit(run_optuna_launch, launch_fname): launch_name})
                    LOG(f'{idle_runners_af.v:.1f} idle runners exist, submitted launch "{launch_name}"; running launches={len(futures)}')
                    idle_runners_af.reset()
                    
                is_first_time = False

            try:
                while futures:
                    completed_futures, _ = cf.wait(futures, timeout=0.1, return_when=cf.FIRST_COMPLETED)

                    if not completed_futures:
                        break
                        
                    for completed_future in completed_futures:
                        launch_name = futures[completed_future]
                        del futures[completed_future]

                        exc = completed_future.exception()
                        
                        if exc is not None:
                            LOG(f'Launch "{launch_name}" failed: {exc}')
                        else:
                            LOG(f'Launch "{launch_name}" completed')
    
                    if completed_futures:
                        completed_launches_count += len(completed_futures)
                        LOG(f'{completed_launches_count} (+{len(completed_futures)}) launches completed; running launches={len(futures)}')
            except TimeoutError as e:
                pass

            time.sleep(5)

[I 2026-10-05 12:16:41,567] Using an existing study with name '18n_study_31.1' instead of creating a new one.


2026.10.05-12:16:41.897860     0.267 >> Model instance registered, version=528
2026.10.05-12:16:41.911221     0.005 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07-launch528.ipynb"
2026.10.05-12:16:41.911454     0.005 >> 6.0 idle runners exist, submitted launch "18n_ppo_tr_frostbite_07:528"; running launches=1
2026.10.05-12:17:13.259523     0.283 >> Model instance registered, version=529
2026.10.05-12:17:13.271430     0.004 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07-launch529.ipynb"
2026.10.05-12:17:13.271696     0.004 >> 5.7 idle runners exist, submitted launch "18n_ppo_tr_frostbite_07:529"; running launches=2
2026.10.05-12:17:44.386591     0.264 >> Model instance registered, version=530
2026.10.05-12:17:44.395194     0.004 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07-launch530.ipynb"
2026.10.05-12:17:44.395383     0.004 >> 4.5 idle runners exist, submitted launch "18n_ppo_tr_frostbite_07

In [11]:
study = optuna.create_study(
    study_name=optuna_study_name,
    storage=JournalStorage(JournalFileBackend(file_path=optuna_study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs.get('MODEL_VERSION', 'n/a')}')
    
    LOG('\tParams: ')
    
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    LOG(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        LOG(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        LOG(f"\tnumber: {trial.number}")
        LOG(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        LOG(f"\tparams: {trial.params}")
        LOG(f"\tvalues: {trial.values}")

[I 2026-10-05 11:41:38,792] Using an existing study with name '18n_study_31.1' instead of creating a new one.


2026.10.05-11:41:38.798903     5.044 >> Study statistics: 
2026.10.05-11:41:38.800655     0.002 >> 	Number of finished trials: 18
2026.10.05-11:41:38.801364     0.001 >> 	Number of pruned trials: 0
2026.10.05-11:41:38.801954     0.001 >> 	Number of complete trials: 18
2026.10.05-11:41:38.803389     0.001 >> Best trial:
2026.10.05-11:41:38.805040     0.002 >> 	Value: 0.796875
2026.10.05-11:41:38.806121     0.001 >> 	Model version: 521
2026.10.05-11:41:38.806758     0.001 >> 	Params: 
2026.10.05-11:41:38.807248     0.000 >> 		parent: 18d_world_model_09:183


In [12]:
top_trials = sorted(filter(lambda t: t.values, study.trials), key=lambda t: -t.values[0])[:3]
LOG(f'Top trials:')

for i, trial in enumerate(top_trials):
    LOG(f'{i+1}) {CONFIG.target_notebook_name + ':' + trial.user_attrs['MODEL_VERSION']}, value={trial.values[0]}')

2026.10.05-11:41:38.819130     0.012 >> Top trials:
2026.10.05-11:41:38.821645     0.003 >> 1) 18n_ppo_tr_frostbite_07:521, value=0.796875
2026.10.05-11:41:38.822711     0.001 >> 2) 18n_ppo_tr_frostbite_07:525, value=0.7875
2026.10.05-11:41:38.823342     0.001 >> 3) 18n_ppo_tr_frostbite_07:522, value=0.7


In [13]:
values = []

for trial in study.trials:
    if trial.values is not None:
        values.append(trial.values[0])

values = np.array(values)
values.mean(), values.std()

(np.float64(0.6157986111111111), np.float64(0.08619105412147472))

In [14]:
','.join(map(str, values.tolist()))

'0.603125,0.6,0.6,0.63125,0.540625,0.671875,0.496875,0.53125,0.571875,0.6,0.659375,0.796875,0.7,0.53125,0.496875,0.7875,0.571875,0.69375'